In [ ]:
"""
Task 2: Restaurant Recommendation (content-based filtering)

Given user preferences (cuisine, city, price range, max cost, delivery flag,
min rating), score every restaurant by similarity to those preferences and
return the top-N matches.
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

DATA_PATH = Path(r"D:\Code\Repo\Machine-Learning-Internship\Dataset .csv")


# ---------- Preprocessing ----------


In [ ]:
def load_clean(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["Cuisines"] = df["Cuisines"].fillna("Unknown")
    for col in ["City", "Currency", "Has Online delivery", "Has Table booking"]:
        df[col] = df[col].fillna("Unknown")
    for col in ["Average Cost for two", "Price range", "Aggregate rating", "Votes"]:
        df[col] = df[col].fillna(df[col].median())
    return df.reset_index(drop=True)


# ---------- Feature building ----------


In [ ]:
def build_features(df: pd.DataFrame):
    """
    Content profile for each restaurant:
      - TF-IDF of comma-separated cuisines (captures similarity between e.g.
        'Italian, Pizza' and 'Pizza, Fast Food')
      - Normalized numeric features: price range, cost, rating
    """
    tfidf = TfidfVectorizer(token_pattern=r"[A-Za-z]+")
    cuisine_matrix = tfidf.fit_transform(
        df["Cuisines"].str.replace(",", " ").str.lower()
    )

    scaler = MinMaxScaler()
    numeric = scaler.fit_transform(
        df[["Price range", "Average Cost for two", "Aggregate rating"]].values
    )
    return tfidf, cuisine_matrix, scaler, numeric


In [ ]:
def build_user_vector(user_pref, tfidf, scaler, df):
    """Project user preferences into the same feature spaces as restaurants."""
    cuisine_vec = tfidf.transform([user_pref["cuisines"].lower()])

    # Fill missing numeric prefs with dataset medians so they're neutral.
    price = user_pref.get("price_range", df["Price range"].median())
    cost = user_pref.get("max_cost", df["Average Cost for two"].median())
    rating = user_pref.get("min_rating", 3.5)
    numeric_vec = scaler.transform([[price, cost, rating]])
    return cuisine_vec, numeric_vec


# ---------- Scoring ----------


In [ ]:
def recommend(df, user_pref, tfidf, cuisine_matrix, scaler, numeric,
              top_n=5, cuisine_weight=0.7, numeric_weight=0.3):
    # Hard filters first — a recommendation that fails a stated constraint
    # is almost never useful, no matter how high its similarity score.
    mask = pd.Series(True, index=df.index)
    if user_pref.get("city"):
        mask &= df["City"].str.lower() == user_pref["city"].lower()
    if user_pref.get("online_delivery"):
        mask &= df["Has Online delivery"] == "Yes"
    if user_pref.get("min_rating") is not None:
        mask &= df["Aggregate rating"] >= user_pref["min_rating"]
    if user_pref.get("max_cost") is not None:
        mask &= df["Average Cost for two"] <= user_pref["max_cost"]

    candidates = df[mask]
    if candidates.empty:
        return candidates

    cand_idx = candidates.index
    cuisine_vec, numeric_vec = build_user_vector(user_pref, tfidf, scaler, df)

    cuisine_sim = cosine_similarity(cuisine_vec, cuisine_matrix[cand_idx]).ravel()
    # Numeric similarity = 1 - normalized L2 distance (clipped to [0,1])
    num_dist = np.linalg.norm(numeric[cand_idx] - numeric_vec, axis=1)
    numeric_sim = 1 - (num_dist / np.sqrt(numeric.shape[1]))
    numeric_sim = np.clip(numeric_sim, 0, 1)

    score = cuisine_weight * cuisine_sim + numeric_weight * numeric_sim
    out = candidates.assign(
        cuisine_sim=cuisine_sim, numeric_sim=numeric_sim, score=score
    )
    return out.sort_values("score", ascending=False).head(top_n)


In [ ]:
def show(title, recs):
    print(f"\n=== {title} ===")
    if recs.empty:
        print("  (no restaurants match the hard filters)")
        return
    cols = [
        "Restaurant Name", "City", "Cuisines",
        "Average Cost for two", "Price range",
        "Aggregate rating", "Votes", "score",
    ]
    print(recs[cols].to_string(index=False))


# ---------- Demo ----------


In [ ]:
def main():
    df = load_clean(DATA_PATH)
    print(f"Loaded {len(df)} restaurants")
    tfidf, cuisine_matrix, scaler, numeric = build_features(df)

    tests = [
        ("User A — New Delhi, North Indian, budget", {
            "cuisines": "North Indian, Mughlai",
            "city": "New Delhi",
            "price_range": 2,
            "max_cost": 800,
            "min_rating": 3.5,
        }),
        ("User B — New Delhi, Italian/Pizza, mid-range + online delivery", {
            "cuisines": "Italian, Pizza",
            "city": "New Delhi",
            "price_range": 3,
            "max_cost": 1500,
            "min_rating": 4.0,
            "online_delivery": True,
        }),
        ("User C — any city, Japanese/Sushi, premium", {
            "cuisines": "Japanese, Sushi",
            "price_range": 4,
            "max_cost": 5000,
            "min_rating": 4.0,
        }),
        ("User D — Bangalore, Cafe/Desserts, casual", {
            "cuisines": "Cafe, Desserts, Bakery",
            "city": "Bangalore",
            "price_range": 2,
            "max_cost": 1000,
            "min_rating": 4.0,
        }),
    ]

    for title, pref in tests:
        recs = recommend(df, pref, tfidf, cuisine_matrix, scaler, numeric, top_n=5)
        show(title, recs)

    # Simple quality check: for a given cuisine preference, what fraction of
    # the top-N recommendations actually contain that cuisine?
    print("\n=== Quality check: cuisine overlap in top-5 ===")
    for title, pref in tests:
        recs = recommend(df, pref, tfidf, cuisine_matrix, scaler, numeric, top_n=5)
        if recs.empty:
            print(f"  {title}: no matches")
            continue
        wanted = {c.strip().lower() for c in pref["cuisines"].split(",")}
        hits = recs["Cuisines"].apply(
            lambda s: any(w in s.lower() for w in wanted)
        ).sum()
        print(f"  {title}: {hits}/{len(recs)} contain a requested cuisine, "
              f"avg rating={recs['Aggregate rating'].mean():.2f}")


In [ ]:
if __name__ == "__main__":
    main()
